AuthorityEffect: None.


## 3. Sealed Packets: Tamper-Evident Communication

In multi-agent systems and LLM pipelines, an agent might attempt to rewrite the user prompt, omit constraints, or forge execution receipts.

The Symbol Game resolves this with **Sealed Packets**:
- Any prompt, candidate response, or evaluation receipt is serialized to canonical JSON (sorted keys, compact delimiters).
- The payload is hashed with SHA-256 to produce an immutable `digest`.
- If an attacker or buggy agent alters even a single bit (e.g. changing `"difference": 1` to `"difference": 0`), the cryptographic digest breaks, and the host refuses the packet immediately.


In [ ]:
# [Pinned] Environment verification & basic dependencies
import hashlib
import json
import sys

assert sys.version_info >= (3, 10), "Requires Python >= 3.10"
print(f"[OK] Python version: {sys.version.split()[0]}")

In [ ]:
# [Pinned] Sealed Packet Sealing and Verification


def canonical_json_bytes(obj: dict) -> bytes:
    """Produces deterministic UTF-8 bytes with sorted keys and no whitespace."""
    return json.dumps(obj, sort_keys=True, separators=(",", ":")).encode("utf-8")


def seal_packet(payload: dict) -> dict:
    """Wraps a payload into a sealed envelope with a SHA-256 fingerprint."""
    raw_bytes = canonical_json_bytes(payload)
    digest = hashlib.sha256(raw_bytes).hexdigest()
    return {
        "format": "symbol_game_packet_v1",
        "digest": digest,
        "payload": payload,
    }


def verify_packet(envelope: dict) -> bool:
    """Checks whether the envelope payload exactly matches its declared digest."""
    expected_digest = envelope.get("digest")
    payload = envelope.get("payload")
    if not expected_digest or payload is None:
        return False
    actual_digest = hashlib.sha256(canonical_json_bytes(payload)).hexdigest()
    return actual_digest == expected_digest


sample_data = {"agent_id": "novice_agent", "action": "propose_cycle", "edges": 3}
sealed = seal_packet(sample_data)
print(f"Sealed Envelope Digest: {sealed['digest']}")
assert verify_packet(sealed) is True, "Valid packet must verify!"

tampered = json.loads(json.dumps(sealed))
tampered["payload"]["edges"] = 999

is_tampered_valid = verify_packet(tampered)
print(f"Is Tampered Packet Valid? {is_tampered_valid}")
assert is_tampered_valid is False, "Tampered packet must be rejected by host!"
print("[OK] Sealed packet integrity and tamper-refusal verified!")

> **Claim-typing legend**
> - **[Pinned]**: taken from a file this repository already has.
> - **[Proved]**: a finite check in this notebook. Not a general proof.
> - **[Declared]**: a carrier, rule, or heading chosen for this sitting.
> - **[Open]**: something this sitting does not decide.
## Anchors

- Former notebook: `00_symbol_game_for_beginners`
- The code checks one finite example. It is not a general proof.

## Exercise

Change one declared input in the code cell. The check should fail for the wrong object and pass for the finite example written here.
